# Task 2 — Agentic Operations Layer

This notebook builds the operations-team-facing agent on top of the artefacts produced by `complaint_analysis.ipynb` (Task 1). Run Task 1 to completion before running this notebook.

**Pipeline:**

`classify` → `assess_urgency` → `retrieve_similar` → `draft_case_note` → `review_gate` → (`human_review` or `auto_output`)

**Loaded from Task 1:**

- `outputs/df_clf.parquet`
- `outputs/embeddings.npy` 
- `outputs/logreg_classifier.pkl`, `outputs/logreg_tfidf.pkl`, `outputs/meta_feature_cols.pkl`, `outputs/label_encoder.pkl`
- `outputs/lora_adapter/` 
- `outputs/chroma_db/` 

In [ ]:
import sys, os
from pathlib import Path

print(f"Python: {sys.version.split()[0]}")
print(f"Working dir: {Path.cwd()}")

import importlib
required_pkgs = [
    ("pandas",                 "pandas"),
    ("numpy",                  "numpy"),
    ("sklearn",                "scikit-learn"),
    ("joblib",                 "joblib"),
    ("tqdm",                   "tqdm"),
    ("dotenv",                 "python-dotenv"),
    ("torch",                  "torch (CUDA wheel for LoRA inference)"),
    ("transformers",           "transformers"),
    ("peft",                   "peft"),
    ("chromadb",               "chromadb"),
    ("langchain_google_genai", "langchain-google-genai"),
    ("langgraph",              "langgraph"),
]
missing = []
for mod, pip_name in required_pkgs:
    try:
        importlib.import_module(mod)
    except ImportError:
        missing.append(pip_name)
if missing:
    print(f"  MISSING packages: {missing}")
    print("  -> pip install -r requirements.txt  +  the torch CUDA wheel from Step 0")
else:
    print("All required packages importable.")

# --- Required Task 1 artefacts ---
OUTPUT_DIR = './outputs'
required_files = [
    f'{OUTPUT_DIR}/df_clf.parquet',
    f'{OUTPUT_DIR}/embeddings.npy',
    f'{OUTPUT_DIR}/logreg_classifier.pkl',
    f'{OUTPUT_DIR}/logreg_tfidf.pkl',
    f'{OUTPUT_DIR}/meta_feature_cols.pkl',
    f'{OUTPUT_DIR}/label_encoder.pkl',
]
missing_files = [f for f in required_files if not Path(f).exists()]
if missing_files:
    print("\n  MISSING Task 1 artefacts:")
    for f in missing_files:
        print(f"    - {f}")
    print("  -> run complaint_analysis.ipynb to completion (especially the save-trailer cell)")
else:
    print("All required Task 1 artefacts present.")

from dotenv import load_dotenv
load_dotenv()
gem_key = os.getenv("GOOGLE_API_KEY") or os.getenv("GEMINI_API_KEY")
print(f"Gemini API key: {'set (' + str(len(gem_key)) + ' chars)' if gem_key else 'NOT set in .env'}")

import torch
print(f"torch: {torch.__version__} | CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"  GPU: {torch.cuda.get_device_name(0)}")


Python: 3.13.13
Working dir: d:\Work\CocaColaHBC\Financial_Complaints_Analyzer


d:\Work\CocaColaHBC\Financial_Complaints_Analyzer\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


All required packages importable.
All required Task 1 artefacts present.
Gemini API key: set (53 chars)
torch: 2.7.1+cu118 | CUDA available: True
  GPU: NVIDIA GeForce RTX 2060


## Setup — Imports, config, and load Task 1 artefacts


In [3]:
import os, json, re, sys
from pathlib import Path
import numpy as np
import pandas as pd
import joblib
from tqdm import tqdm
from dotenv import load_dotenv
from sklearn.model_selection import train_test_split

load_dotenv()

if not os.getenv("GOOGLE_API_KEY") and os.getenv("GEMINI_API_KEY"):
    os.environ["GOOGLE_API_KEY"] = os.environ["GEMINI_API_KEY"]

import torch
print(f"torch: {torch.__version__} | CUDA: {torch.cuda.is_available()}")


torch: 2.7.1+cu118 | CUDA: True


In [4]:
# Config
OUTPUT_DIR              = './outputs'
RANDOM_STATE            = 42
EMBEDDING_MODEL_NAME    = 'sentence-transformers/all-MiniLM-L6-v2'

# Classifier choice
CLASSIFIER_TO_USE       = 'logreg'     
CONFIDENCE_THRESHOLD    = 0.60         
MAX_LENGTH              = 256        

# LoRA inference paths
LORA_ADAPTER_PATH       = f'{OUTPUT_DIR}/lora_adapter'
DEVICE                  = 'cuda' if torch.cuda.is_available() else 'cpu'

# LLM
GEMINI_MODEL            = 'gemini-3.1-flash-lite'  

# Vector store
CHROMA_PATH             = f'{OUTPUT_DIR}/chroma_db'

print(f"CLASSIFIER_TO_USE = {CLASSIFIER_TO_USE!r} | DEVICE = {DEVICE} | CHROMA_PATH = {CHROMA_PATH}")


CLASSIFIER_TO_USE = 'logreg' | DEVICE = cuda | CHROMA_PATH = ./outputs/chroma_db


In [ ]:
# Load Task 1 artefacts
import re
from sentence_transformers import SentenceTransformer

df_clf = pd.read_parquet(f'{OUTPUT_DIR}/df_clf.parquet')
embeddings = np.load(f'{OUTPUT_DIR}/embeddings.npy')

# After post-hoc outlier reduction in Task 1, df_clf contains every embedded row
# (no topic -1 left). If a future re-fit reintroduces noise, filter here before
# slicing clf_embeddings.
assert len(df_clf) == len(embeddings), (
    f"df_clf rows ({len(df_clf)}) must match embeddings ({len(embeddings)}). "
    f"Re-run the Task 1 save cell and the embeddings cache."
)
clf_embeddings = embeddings

# Sentence-transformer for embedding NEW complaints at inference (used by
# predict_logreg and find_similar). The same model that produced embeddings.npy.
embedding_model = SentenceTransformer(EMBEDDING_MODEL_NAME)

# Text cleaning function — copied verbatim from Task 1 cell 13. Must stay in
# lockstep with the function used at training time, otherwise new complaints
# get a different normalisation than the training set and accuracy drops
# silently.
def clean_text(text: str) -> str:
    if not isinstance(text, str):
        return ''
    text = re.sub(r'http\S+|www\S+', '', text)  # remove URLs
    text = re.sub(r'\s+', ' ', text).strip()     # normalise whitespace
    return text.lower()

print(f"df_clf: {df_clf.shape} | {df_clf['topic_label'].nunique()} topic labels")
print(f"clf_embeddings: {clf_embeddings.shape}")
print(f"embedding_model: {EMBEDDING_MODEL_NAME}")
print(f"clean_text() defined.")


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 6025.04it/s]


df_clf: (20998, 22) | 25 topic labels
clf_embeddings: (20998, 384)
embedding_model: sentence-transformers/all-MiniLM-L6-v2
clean_text() defined.


## Section 11 — Vector Database (ChromaDB)
==============================================================================


In [ ]:
import chromadb

CHROMA_PATH = f'{OUTPUT_DIR}/chroma_db'

chroma_client = chromadb.PersistentClient(path=CHROMA_PATH)

try:
    chroma_client.delete_collection("complaints")
except:
    pass
collection = chroma_client.create_collection(
    name="complaints",
    metadata={"hnsw:space": "cosine"}  # Use cosine similarity for normalised embeddings
)

# ── Demo set = Task 1 test set (held out from classifier training) ────────────
# We deliberately use idx_test (saved by Task 1 cell 26) so the demo complaints
# are unseen by BOTH the classifier and ChromaDB. The ChromaDB index then covers
# train+val only — realistic operational scenario: retrieval surfaces complaints
# the agent has "seen before", and the classifier has trained on them, but the
# *new* complaint coming in (the demo) is genuinely novel to both components.
idx_test = np.load(f'{OUTPUT_DIR}/idx_test.npy')
idx_all  = np.arange(len(df_clf))
idx_index = np.setdiff1d(idx_all, idx_test, assume_unique=True)

df_index = df_clf.iloc[idx_index].reset_index(drop=True)
emb_index = clf_embeddings[idx_index]
df_demo   = df_clf.iloc[idx_test].reset_index(drop=True)

print(f"ChromaDB index size:  {len(df_index):,}  (= Task 1 train + val; classifier was trained on this)")
print(f"Demo set size:        {len(df_demo):,}  (= Task 1 test; unseen by classifier AND ChromaDB)")
print(f"Intersection check:   {len(set(idx_index) & set(idx_test))} (should be 0)")

CHROMA_BATCH = 500
for start in tqdm(range(0, len(df_index), CHROMA_BATCH)):
    end = min(start + CHROMA_BATCH, len(df_index))
    batch_df = df_index.iloc[start:end]
    batch_emb = emb_index[start:end]

    collection.add(
        embeddings=batch_emb.tolist(),
        documents=batch_df['complaint_what_happened'].fillna('').tolist(),
        metadatas=[{
            "topic_label":    row['topic_label'],
            "company_response": str(row.get('company_response', 'Unknown')),
            "product":        str(row.get('product', 'Unknown')),
            "sub_product":    str(row.get('sub_product', 'Unknown')),
            "timely":         str(row.get('timely', 'Unknown')),
            "complaint_id":   str(row.get('complaint_id', f'ID_{i}')),
        } for i, (_, row) in enumerate(batch_df.iterrows())],
        ids=[f"doc_{start + i}" for i in range(end - start)]
    )

print(f" ChromaDB index built: {collection.count():,} entries → {CHROMA_PATH}")

df_demo.to_parquet(f'{OUTPUT_DIR}/demo_complaints.parquet', index=False)
print(f"Demo set saved → {OUTPUT_DIR}/demo_complaints.parquet")

sample_demo_emb = clf_embeddings[idx_test[0]].tolist()
results = collection.query(query_embeddings=[sample_demo_emb], n_results=3)
print("\nSanity check — top 3 similar complaints for the first demo complaint:")
for doc, meta in zip(results['documents'][0], results['metadatas'][0]):
    print(f"  [{meta['topic_label']}] {meta['company_response']} | {doc[:100]}...")



ChromaDB index size:  16,798  (= Task 1 train + val; classifier was trained on this)
Demo set size:        4,200  (= Task 1 test; unseen by classifier AND ChromaDB)
Intersection check:   0 (should be 0)


100%|██████████| 34/34 [03:56<00:00,  6.96s/it]

 ChromaDB index built: 16,798 entries → ./outputs/chroma_db
Demo set saved → ./outputs/demo_complaints.parquet

Sanity check — top 3 similar complaints for the first demo complaint:
  [General Account & Customer Service Issues] Closed with explanation | On XX/XX/2020 I purchase an airline ticket from XXXX XXXX with Chase credit card for the amount of {...
  [General Account & Customer Service Issues] Closed with explanation | On XX/XX/2019, I booked a flight through the merchant - XXXX XXXX using my Chase credit card. I had ...
  [General Account & Customer Service Issues] Closed with monetary relief | To Whom It May Concern, I purchased an airline ticket with XXXX on XX/XX/2020 using my Chase Visa cr...


## Section 12 — LangGraph Agent

Architecture:
  classify → assess_urgency → retrieve_similar → draft_case_note
      → review_gate → [human_review | auto_output]

Design principles:
- Local classifier (no API cost) handles topic assignment
- Rule-based urgency checks run before LLM calls (saves tokens)
- LLM (Gemini) only called for ambiguous urgency and case note drafting
- Confidence < threshold OR urgency = HIGH → mandatory human review
- All LLM outputs grounded in: classifier topic, metadata rules, ChromaDB precedents
  (this addresses the bonus criterion about preventing hallucination)

==============================================================================


In [ ]:
# Load all inference artifacts
import joblib, json, re
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.messages import HumanMessage

# Label encoder
le = joblib.load(f'{OUTPUT_DIR}/label_encoder.pkl')
NUM_CLASSES = len(le.classes_)

# LogReg artifacts
clf_logreg = joblib.load(f'{OUTPUT_DIR}/logreg_classifier.pkl')
tfidf      = joblib.load(f'{OUTPUT_DIR}/logreg_tfidf.pkl')
META_COLS  = joblib.load(f'{OUTPUT_DIR}/meta_feature_cols.pkl')

# LoRA artifacts
from transformers import DistilBertTokenizerFast, DistilBertForSequenceClassification
from peft import PeftModel

if CLASSIFIER_TO_USE == 'lora':
    if 'lora_model' not in dir():
        _base = DistilBertForSequenceClassification.from_pretrained(
            'distilbert-base-uncased', num_labels=NUM_CLASSES
        )
        lora_model = PeftModel.from_pretrained(_base, LORA_ADAPTER_PATH)
        tokenizer  = DistilBertTokenizerFast.from_pretrained(LORA_ADAPTER_PATH)
        lora_model.eval().to(DEVICE)

# ChromaDB
chroma_client = chromadb.PersistentClient(path=CHROMA_PATH)
collection     = chroma_client.get_collection("complaints")

llm = ChatGoogleGenerativeAI(
    model=GEMINI_MODEL,
    temperature=0.1,     
                        
    google_api_key=os.getenv("GOOGLE_API_KEY"),
    max_retries=2
)

def _text_of(resp):
    """Gemini may return resp.content as a string OR a list of message parts.
    Return a plain string either way."""
    c = resp.content
    if isinstance(c, str):
        return c
    return ''.join(
        p.get('text', '') if isinstance(p, dict) else str(p) for p in c
    )

print(" All inference artifacts loaded.")

 All inference artifacts loaded.


In [ ]:
def predict_logreg(text: str, product: str = 'Unknown', submitted_via: str = 'Unknown') -> tuple:
    """Returns (topic_label, confidence) using the Logistic Regression classifier."""
    from scipy.sparse import hstack, csr_matrix
    clean = clean_text(text)
    X_t = tfidf.transform([clean])
    X_e = csr_matrix(embedding_model.encode([clean]))
    meta = pd.get_dummies(
        pd.DataFrame([{'product': product, 'submitted_via': submitted_via, 'timely': 'Unknown'}])
    ).reindex(columns=META_COLS, fill_value=0)
    X = hstack([X_t, X_e, csr_matrix(meta.values.astype(float))])
    proba = clf_logreg.predict_proba(X)[0]
    return le.inverse_transform([proba.argmax()])[0], float(proba.max())

def predict_lora_single(text: str) -> tuple:
    """Returns (topic_label, confidence) using the LoRA fine-tuned DistilBERT."""
    enc = tokenizer(clean_text(text), truncation=True, max_length=MAX_LENGTH,
                    padding=True, return_tensors='pt')
    enc = {k: v.to(DEVICE) for k, v in enc.items()}
    with torch.no_grad():
        logits = lora_model(**enc).logits
    proba = torch.softmax(logits, dim=-1)[0].cpu().numpy()
    return le.inverse_transform([proba.argmax()])[0], float(proba.max())

def find_similar(text: str, topic_label: str, n: int = 3) -> list:
    """Semantic search in ChromaDB. Filters by matching topic for relevance."""
    emb = embedding_model.encode([clean_text(text)], normalize_embeddings=True).tolist()
    try:
        results = collection.query(
            query_embeddings=emb,
            n_results=n,
            where={"topic_label": topic_label}  # same topic only
        )
    except Exception:
        # Fallback: search without topic filter if topic has too few indexed docs
        results = collection.query(query_embeddings=emb, n_results=n)

    similar = []
    for doc, meta in zip(results['documents'][0], results['metadatas'][0]):
        similar.append({
            "text_excerpt": doc[:250],
            "topic":        meta.get('topic_label', 'Unknown'),
            "outcome":      meta.get('company_response', 'Unknown'),
            "product":      meta.get('product', 'Unknown'),
        })
    return similar

print(" Inference functions defined.")

 Inference functions defined.


In [ ]:
# Define LangGraph agent
from typing import TypedDict, Optional, List
from langgraph.graph import StateGraph, END
from langgraph.checkpoint.memory import MemorySaver

class ComplaintState(TypedDict):
    complaint_text:   str
    product:          str
    sub_product:      str
    tags:             str          
    submitted_via:    str
    timely:           str
    topic_label:      str
    confidence:       float
    urgency:          str          
    urgency_reason:   str
    similar_cases:    list
    case_note:        str
    recommended_team: str
    needs_human_review: bool
    human_review_reason: str

# ── Routing map (one per topic in TOPIC_LABELS) ─────────────────────────────
ROUTING_MAP = {
    "General Account & Customer Service Issues": "Customer Service Team",

    # Checking / savings operations
    "Overdraft & Insufficient Funds Fees":       "Retail Banking Operations",
    "Account Opening Bonus Disputes":            "Retail Banking Operations",

    # Credit card — billing, rates, rewards
    "Credit Card Rewards & Points Disputes":     "Card Rewards Team",
    "Credit Card Limit / Score / Reporting Issues": "Card Services Team",
    "Credit Card Balance & Payment Disputes":    "Card Billing Operations",
    "Credit Card APR / Rate Change Disputes":    "Card Pricing Team",
    "Credit Card Late Fee Disputes":             "Card Billing Operations",
    "Co-Branded Retail Card Issues":                  "Co-Branded Cards Team",

    # Credit reporting / inquiries
    "Unauthorized Hard Credit Inquiries":        "Credit Disputes Team",
    "Disputed Late Payment Reporting":           "Credit Disputes Team",
    "Bank Credit Report Hard Inquiries":        "Credit Disputes Team",
    "Identity Theft / FCRA Disputes":            "Fraud Investigation Team",

    # Mortgage
    "Mortgage Loan Modification Requests":       "Loss Mitigation Team",
    "Mortgage Escrow / Property Tax Issues":     "Mortgage Escrow Team",
    "Mortgage Refinance / Rate Disputes":        "Mortgage Originations Team",
    "Mortgage Short Sale Issues":                "Loss Mitigation Team",
    "Mortgage Payment Processing":               "Mortgage Servicing Team",
    "Formal/Legal Mortgage Disputes": "Mortgage Legal Operations",

    # Debt collection
    "Credit Card Debt Collection Disputes":      "Collections Compliance Team",
    "Debt Collection Phone Calls":               "Collections Compliance Team",

    # Vulnerable populations
    "Senior / POA Account Access Issues":        "Vulnerable Customer Team",

    # Payments / transfers
    "Money Transfer / Wire Disputes":            "Payments Operations Team",

    # Auto lending
    "Auto Loan Title / Lien Issues":             "Auto Lending Team",

    # Bankruptcy / legal
    "Bankruptcy & Credit Reporting Disputes":    "Bankruptcy & Legal Operations",

    # Safety net for any future re-fit that produces noise
    "Uncategorised":                             "Customer Service Team",
}

# High-risk keyword patterns → instant HIGH urgency (no LLM token cost)
HIGH_URGENCY_PATTERNS = [
    r'\b(attorney|lawyer|lawsuit|sue|court|legal action)\b',
    r'\b(cfpb|consumer financial protection bureau)\b',
    r'\bidentity theft\b',
    r'\bfraud\b',
    r'\b(foreclosure|eviction|repossession|repo)\b',
    r'\b(bankruptcy|chapter 7|chapter 11|chapter 13)\b',
    r'\bsuicid',    # distress signals — truncated to match variations
    r'\bhospital\b',
]

def classify_node(state: ComplaintState) -> ComplaintState:
    """Uses local classifier — zero API tokens consumed here."""
    if CLASSIFIER_TO_USE == 'lora':
        label, conf = predict_lora_single(state['complaint_text'])
    else:
        label, conf = predict_logreg(
            state['complaint_text'],
            state.get('product', 'Unknown'),
            state.get('submitted_via', 'Unknown')
        )
    return {**state, "topic_label": label, "confidence": conf}

def assess_urgency_node(state: ComplaintState) -> ComplaintState:
    """
    Hybrid urgency assessment:
    1. Rule-based (fast, free) → catches obvious HIGH cases
    2. Gemini (only for ambiguous cases) → nuanced reasoning
    """
    text_lower = state['complaint_text'].lower()

    # Rule 1: high-risk keywords
    for pattern in HIGH_URGENCY_PATTERNS:
        if re.search(pattern, text_lower):
            return {**state, "urgency": "HIGH",
                    "urgency_reason": f"High-risk keyword pattern detected: '{pattern}'"}

    # Rule 2: vulnerability tag
    if state.get('tags', '') in ['Servicemember', 'Older American']:
        return {**state, "urgency": "HIGH",
                "urgency_reason": f"Vulnerable customer flag: {state['tags']}"}

    # Rule 3: untimely response — elevates base urgency
    timely_flag = state.get('timely', 'Yes') == 'No'

    # Gemini for everything else (medium/low distinction requires context)
    prompt = f"""You are a financial services compliance officer. Assess the urgency of this customer complaint.

Complaint (first 500 chars): {state['complaint_text'][:500]}
Classified topic: {state['topic_label']} (confidence: {state['confidence']:.0%})
Product: {state.get('product', 'Unknown')}
Untimely response on record: {timely_flag}

Rate urgency as HIGH, MEDIUM, or LOW.
HIGH = regulatory risk, significant financial harm, time-sensitive legal issue, vulnerable customer
MEDIUM = clear financial impact but not immediately escalatory
LOW = informational, minor inconvenience, no immediate harm

Respond with ONLY this JSON (no markdown):
{{"urgency": "HIGH" or "MEDIUM" or "LOW", "reason": "one concise sentence"}}"""

    try:
        resp = llm.invoke([HumanMessage(content=prompt)])
        raw = _text_of(resp).strip().strip('```json').strip('```').strip()
        result = json.loads(raw)
        urgency = result.get('urgency', 'MEDIUM')
        reason  = result.get('reason', 'LLM assessment')
        if timely_flag and urgency == 'LOW':
            urgency = 'MEDIUM'
            reason += ' (elevated: untimely response on record)'
    except Exception as e:
        urgency, reason = 'MEDIUM', f'Urgency parsing failed ({e}) — defaulting to MEDIUM'

    return {**state, "urgency": urgency, "urgency_reason": reason}

def retrieve_similar_node(state: ComplaintState) -> ComplaintState:
    """Semantic search — no API tokens consumed."""
    similar = find_similar(state['complaint_text'], state['topic_label'], n=3)
    return {**state, "similar_cases": similar}

def draft_case_note_node(state: ComplaintState) -> ComplaintState:
    """
    Gemini drafts the case note, grounded in:
    - Local classifier output (topic + confidence) — prevents hallucinated categorisation
    - Urgency assessment — ensures risk framing is data-driven
    - Real historical resolutions from ChromaDB — prevents hallucinated outcomes

    The ops agent reviews and edits this draft before it enters the case system.
    """
    similar_text = "\n".join([
        f"  • [{c['topic']}] Resolved as '{c['outcome']}' (Product: {c['product']})\n"
        f"    Excerpt: {c['text_excerpt']}"
        for c in state['similar_cases']
    ]) if state['similar_cases'] else "  No similar historical cases found."

    team = ROUTING_MAP.get(state['topic_label'], "General Operations Queue")

    prompt = f"""You are a financial services operations analyst writing an internal case note.
This note will be reviewed by a human agent before any action is taken.

━━━ COMPLAINT ━━━
{state['complaint_text'][:700]}

━━━ CLASSIFICATION (AI-generated, pending human review) ━━━
Topic: {state['topic_label']} | Confidence: {state['confidence']:.0%}
Product: {state.get('product', 'Unknown')} / {state.get('sub_product', 'Unknown')}
Urgency: {state['urgency']} — {state['urgency_reason']}
Submitted via: {state.get('submitted_via', 'Unknown')}
Vulnerability tag: {state.get('tags', 'None')}

━━━ SIMILAR HISTORICAL CASES (from complaint database) ━━━
{similar_text}

Write an internal case note with these four sections:
**SUMMARY** (2 sentences max — what the customer is complaining about)
**KEY ISSUES** (3-5 bullet points — specific claims or regulatory concerns)
**RECOMMENDED ACTION** (based on similar historical resolutions above — cite them)
**RISK FLAGS** (regulatory exposure, escalation risk, or vulnerability concerns)

Be factual. Do not invent resolutions. If uncertain, state it explicitly."""

    try:
        resp = llm.invoke([HumanMessage(content=prompt)])
        case_note = _text_of(resp)
    except Exception as e:
        case_note = f"[Case note generation failed: {e}. Manual review required.]"

    return {**state, "case_note": case_note, "recommended_team": team}

def review_gate_node(state: ComplaintState) -> ComplaintState:
    """
    Three hard stops for mandatory human review:
    1. Low classifier confidence (< threshold)
    2. HIGH urgency — requires manager sign-off
    3. No similar historical cases — novel complaint type, no precedent

    This is the core human-in-the-loop mechanism.
    The agent NEVER autonomously routes a case that triggers any of these.
    """
    reasons = []
    if state['confidence'] < CONFIDENCE_THRESHOLD:
        reasons.append(f"Low confidence ({state['confidence']:.0%} < {CONFIDENCE_THRESHOLD:.0%})")
    if state['urgency'] == 'HIGH':
        reasons.append("HIGH urgency — manager approval required before routing")
    if not state['similar_cases']:
        reasons.append("No historical precedent — novel complaint type")

    return {
        **state,
        "needs_human_review": bool(reasons),
        "human_review_reason": " | ".join(reasons) if reasons else ""
    }

def route_after_gate(state: ComplaintState) -> str:
    return "human_review" if state['needs_human_review'] else "auto_output"

def human_review_node(state: ComplaintState) -> ComplaintState:
    """
    In production: creates a Jira/ServiceNow ticket and halts.
    Human agent reviews the full packet, overrides topic/urgency if needed,
    then re-triggers the agent with corrected inputs.
    For the prototype: logs the flag and passes the packet through for inspection.
    """
    return state

def auto_output_node(state: ComplaintState) -> ComplaintState:
    """High-confidence, non-urgent complaints are auto-routed."""
    return state

# ── Build and compile the graph ────────────────────────────────────────────────
builder = StateGraph(ComplaintState)

builder.add_node("classify",        classify_node)
builder.add_node("assess_urgency",  assess_urgency_node)
builder.add_node("retrieve_similar",retrieve_similar_node)
builder.add_node("draft_case_note", draft_case_note_node)
builder.add_node("review_gate",     review_gate_node)
builder.add_node("human_review",    human_review_node)
builder.add_node("auto_output",     auto_output_node)

builder.set_entry_point("classify")
builder.add_edge("classify",         "assess_urgency")
builder.add_edge("assess_urgency",   "retrieve_similar")
builder.add_edge("retrieve_similar", "draft_case_note")
builder.add_edge("draft_case_note",  "review_gate")
builder.add_conditional_edges("review_gate", route_after_gate,
    {"human_review": "human_review", "auto_output": "auto_output"})
builder.add_edge("human_review", END)
builder.add_edge("auto_output",  END)

memory = MemorySaver()
graph  = builder.compile(checkpointer=memory)

print(" LangGraph agent compiled.")
print("\nAgent flow:")
print("  classify → assess_urgency → retrieve_similar → draft_case_note")
print("      → review_gate → [human_review | auto_output]")

 LangGraph agent compiled.

Agent flow:
  classify → assess_urgency → retrieve_similar → draft_case_note
      → review_gate → [human_review | auto_output]


## Section 13 — Agent Demo

### We run 5 complaints from the held-out demo set through the full pipeline. These complaints were NEVER seen during classifier training and are NOT in the ChromaDB index — so this is a genuine end-to-end evaluation.
==============================================================================


In [ ]:
def format_triage_packet(state: dict, complaint_num: int) -> str:
    """Formats the full agent output as a readable triage packet."""
    review_flag = (
        f"\n{'─'*62}\n  HUMAN REVIEW REQUIRED\n{state['human_review_reason']}"
        if state['needs_human_review']
        else f"\n{'─'*62}\n AUTO-ROUTED → {state['recommended_team']}"
    )

    similar_str = "\n".join([
        f"  {i+1}. [{c['topic']}] Outcome: {c['outcome']}\n"
        f"     {c['text_excerpt'][:150]}..."
        for i, c in enumerate(state['similar_cases'])
    ]) or "  None found"

    return f"""
{'═'*62}
TRIAGE PACKET #{complaint_num}
{'═'*62}
CLASSIFICATION
  Topic:      {state['topic_label']}
  Confidence: {state['confidence']:.0%}
  Product:    {state.get('product', 'Unknown')}

URGENCY: {state['urgency']}
  {state['urgency_reason']}

SIMILAR HISTORICAL CASES
{similar_str}

CASE NOTE
{state['case_note']}
{review_flag}
{'═'*62}
"""

In [ ]:
df_demo = pd.read_parquet(f'{OUTPUT_DIR}/demo_complaints.parquet')

demo_sample = (
    df_demo.groupby('topic_label', group_keys=False)
           .apply(lambda g: g.sample(1, random_state=RANDOM_STATE))
           .head(5)
           .reset_index(drop=True)
)

print(f"Running {len(demo_sample)} complaints through the agent...\n")

for i, (_, row) in enumerate(demo_sample.iterrows()):
    complaint_input = {
        "complaint_text": row['complaint_what_happened'],
        "product":        str(row.get('product', '')),
        "sub_product":    str(row.get('sub_product', '')),
        "tags":           str(row.get('tags', '')) if pd.notna(row.get('tags')) else '',
        "submitted_via":  str(row.get('submitted_via', '')),
        "timely":         str(row.get('timely', 'Yes')),
    }

    config = {"configurable": {"thread_id": f"demo-{i+1}"}}
    result = graph.invoke(complaint_input, config=config)

    print(format_triage_packet(result, i + 1))

# ==============================================================================

Running 5 complaints through the agent...


══════════════════════════════════════════════════════════════
TRIAGE PACKET #1
══════════════════════════════════════════════════════════════
CLASSIFICATION
  Topic:      Account Opening Bonus Disputes
  Confidence: 86%
  Product:    Checking or savings account

URGENCY: MEDIUM
  The complaint involves a specific financial dispute regarding promotional funds that requires investigation but does not indicate immediate regulatory or legal escalation.

SIMILAR HISTORICAL CASES
  1. [Account Opening Bonus Disputes] Outcome: Closed with explanation
     On XX/XX/XXXX I received an email solicitation from Chase regarding a promotion it was offering through XX/XX/XXXX. I have been a Chase credit card ho...
  2. [Account Opening Bonus Disputes] Outcome: Closed with explanation
     Hello - In XXXX of this year I opened a Checking and Savings Account with JP Morgan Chase ( Chase ) under a {$600.00} promotion offer. The terms of th...
  3. [Account Op

## Section 14 — Daily Operations Briefing

### A batch function that processes a day's worth of new complaints and produces a morning briefing for the team lead. 
==============================================================================


In [ ]:
def generate_daily_briefing(complaints: pd.DataFrame) -> str:
    """
    Classifies a batch of complaints and generates a team lead briefing.
    Uses local classifier only (no LLM calls per complaint) for cost efficiency.
    One LLM call at the end to synthesise the summary.
    """
    results = []
    for _, row in complaints.iterrows():
        text = str(row.get('complaint_what_happened', ''))
        if not text or len(text) < 50:
            continue

        if CLASSIFIER_TO_USE == 'lora':
            label, conf = predict_lora_single(text)
        else:
            label, conf = predict_logreg(text, str(row.get('product', 'Unknown')))

        # Rule-based urgency check only (no LLM cost per complaint)
        is_high = any(re.search(p, text.lower()) for p in HIGH_URGENCY_PATTERNS)
        is_high = is_high or str(row.get('tags', '')) in ['Servicemember', 'Older American']

        results.append({
            "topic":       label,
            "confidence":  conf,
            "urgency":     "HIGH" if is_high else "MEDIUM/LOW",
            "product":     str(row.get('product', 'Unknown')),
            "low_conf":    conf < CONFIDENCE_THRESHOLD,
        })

    if not results:
        return "No complaints to brief."

    res_df = pd.DataFrame(results)
    topic_dist  = res_df['topic'].value_counts().to_dict()
    high_count  = (res_df['urgency'] == 'HIGH').sum()
    review_count= res_df['low_conf'].sum()
    avg_conf    = res_df['confidence'].mean()

    summary_input = f"""
Total complaints processed: {len(res_df)}
Topic distribution: {json.dumps(topic_dist, indent=2)}
HIGH urgency cases: {high_count}
Flagged for human review (low confidence): {review_count}
Average classifier confidence: {avg_conf:.0%}
Top product: {res_df['product'].value_counts().index[0]}
"""

    prompt = f"""You are a financial operations team lead. Write a concise morning briefing based on yesterday's complaint data.

DATA:
{summary_input}

Write the briefing in this format:
**DAILY COMPLAINT BRIEFING — [today's date]**

**Volume & Topics**
(2-3 sentences on volume and dominant topics)

**Action Items**
(bullet points — specific, actionable, prioritised by urgency)

**Watch List**
(any patterns or emerging issues that warrant monitoring)

Keep it under 200 words. Be direct and operational."""

    try:
        resp = llm.invoke([HumanMessage(content=prompt)])
        briefing = _text_of(resp)
    except Exception as e:
        briefing = f"[Briefing generation failed: {e}]\n\nRaw stats:\n{summary_input}"

    return briefing

# Run the daily briefing on the demo set (simulating yesterday's complaints)
print("Generating daily operations briefing...")
briefing = generate_daily_briefing(df_demo.head(50))  # Use 50 complaints for demo speed
print("\n" + "=" * 62)
print(briefing)
print("=" * 62)

Generating daily operations briefing...

**DAILY COMPLAINT BRIEFING — May 22, 2024**

**Volume & Topics**
We processed 50 complaints yesterday, with Credit/Prepaid cards emerging as the primary product category. General Account and Customer Service issues remain the dominant volume driver, accounting for 32% of total intake.

**Action Items**
*   **Prioritize High-Urgency Cases:** Address the 26 flagged high-urgency items immediately to meet SLA requirements.
*   **Manual Review:** Assign the 15 low-confidence cases (70% average confidence) to senior analysts for manual verification before final disposition.
*   **Escalation:** Review the 2 formal/legal mortgage disputes for immediate legal department referral.

**Watch List**
*   **Classifier Performance:** The 70% average confidence score is suboptimal; monitor for potential drift in the automated categorization model.
*   **Mortgage Fragmentation:** While volume is low, the variety of mortgage-related complaints (10 total across 6 s